# Plain MLP: balanced assignment, seed ensembling, per-sensor gain augmentation

Three additions to the signed-log MLP, tested independently on the same forward-chaining folds.

### 1. Balanced assignment
The competition states the test set holds **exactly 600 of each class**, and nothing shipped so
far uses it. On the SVM+MLP blend this was worth **+0.098** macro-F1. Retested here on a plain MLP.
Note this is *not* the prior correction that failed earlier (dividing by the training prior, which
cost 0.05) — it is a hard capacity constraint on the assignment.

### 2. Seed ensembling
Currently 3 seeds in CV, 5 at submission. Pure variance reduction on a small-n, high-variance
problem. Implemented so that **15 seeds are trained once per fold and the 1 / 3 / 5 / 15-seed
results are prefixes of the same runs** — so the comparison is exactly nested, not four separate
noisy experiments.

### 3. Per-sensor gain augmentation
The one drift-specific idea left, and the only one that does **not** assume the drift direction is
predictable from history — the assumption that broke OSC (its subspace captured 91.8% of the
batch-9 shift but only 10.6% of the batch-10 shift).

Sensor ageing is largely a **multiplicative gain change per sensor**. In the raw domain that is
`x -> g_s * x` for sensor `s`. Because the features are already `signed_log`-transformed, and
`log(g·x) = log(g) + log(x)`, a multiplicative gain becomes an **additive per-sensor offset** in
the space the model actually sees. So the augmentation is: during training, add a random offset
drawn per sensor (shared across that sensor's 8 descriptors) to every sample, which teaches the
model to be invariant to arbitrary per-sensor gain — in *any* direction, not a direction estimated
from past batches.

Concentration is excluded from the augmentation: it is a dose, not a sensor reading.

**Reference points, same scheme:** RF 0.7807 / 0.828 large-drift · OSC k=2 0.8477 / 0.828 ·
MLP (signed-log, 3 seeds) 0.8474 / 0.8578 · SVM rbf 0.8604 / 0.8686.

In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}"
      + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "  [no GPU]"))

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_sub = pd.read_csv("data/sample_submission.csv")

FEAT = [f"feat_{i}" for i in range(1, 129)]      # 16 sensors x 8 descriptors, sensor-major
COLS = FEAT + ["concentration"]
N_SENSORS, N_DESC = 16, 8
CLASSES = sorted(train["gas_class"].unique())
K = len(CLASSES)
FOLDS = sorted(train["batch"].unique())[1:]
TEST_QUOTA = 600

_s = StandardScaler().fit(train[FEAT])
_X = _s.transform(train[FEAT])
_c = {b: _X[train["batch"].values == b].mean(0) for b in sorted(train["batch"].unique())}
DRIFT = {b: float(np.linalg.norm(_c[b] - _c[b - 1])) for b in FOLDS}
LARGE_DRIFT = [b for b, s in DRIFT.items() if s >= 5.0]

cnt = pd.crosstab(train["batch"], train["gas_class"])
BALANCEABLE = [b for b in FOLDS if (cnt.loc[b] > 0).all() and cnt.loc[b].min() >= 15]
print(f"\nlarge-drift folds: {[int(b) for b in LARGE_DRIFT]}")
print(f"balanceable folds (all 6 classes, >=15 each): {[int(b) for b in BALANCEABLE]}")

device: cuda  (NVIDIA GeForce RTX 5060 Ti)

large-drift folds: [2, 3, 4, 5, 6, 8]
balanceable folds (all 6 classes, >=15 each): [6, 7, 8, 9]


In [2]:
def signed_log(a):
    return np.sign(a) * np.log1p(np.abs(a))


def prep(fit_df, *apply_dfs):
    """signed-log + standardise, fit on the training fold only.
    Also returns the scaler's per-feature scale, needed to inject the gain augmentation
    in the right units."""
    sc = StandardScaler().fit(signed_log(fit_df[COLS].values))
    out = [sc.transform(signed_log(d[COLS].values)).astype(np.float32)
           for d in (fit_df,) + apply_dfs]
    return out, sc.scale_[:len(FEAT)].astype(np.float32)


class MLP(nn.Module):
    def __init__(self, n_in, width=256, p_drop=0.3, n_out=K):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, width), nn.BatchNorm1d(width), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(width, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, n_out))

    def forward(self, x):
        return self.net(x)


def train_seeds(X_tr, y_tr, X_ap, feat_scale, n_seeds=15, aug_sigma=0.0,
                epochs=80, bs=256, lr=2e-3, wd=1e-4):
    """Returns per-seed probabilities, shape (n_seeds, len(X_ap), K), so that any prefix of
    seeds can be averaged afterwards without retraining."""
    Xt = torch.tensor(X_tr, device=DEVICE)
    Xa = torch.tensor(X_ap, device=DEVICE)
    yt = torch.tensor(y_tr, dtype=torch.long, device=DEVICE)
    scale_t = torch.tensor(feat_scale, device=DEVICE)
    n_feat = len(FEAT)
    probs = np.zeros((n_seeds, len(X_ap), K), dtype=np.float64)

    for s in range(n_seeds):
        torch.manual_seed(s)
        m = MLP(Xt.shape[1]).to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        for _ in range(epochs):
            m.train()
            for idx in torch.randperm(len(Xt), device=DEVICE).split(bs):
                if len(idx) < 2:
                    continue
                xb = Xt[idx]
                if aug_sigma > 0:
                    # multiplicative per-sensor gain == additive offset in signed-log space
                    d = torch.randn(len(idx), N_SENSORS, device=DEVICE) * aug_sigma
                    d = d.repeat_interleave(N_DESC, dim=1)          # (B, 128), shared per sensor
                    xb = xb.clone()
                    xb[:, :n_feat] += d / scale_t                    # into standardised units
                loss = F.cross_entropy(m(xb), yt[idx])
                opt.zero_grad(); loss.backward(); opt.step()
            sch.step()
        m.eval()
        with torch.no_grad():
            probs[s] = F.softmax(m(Xa), dim=1).cpu().numpy()
    return probs


def f1_of(P, y):
    return f1_score(y, np.array(CLASSES)[P.argmax(1)], average="macro")

In [3]:
# Train 15 seeds per fold ONCE at aug=0. Every seed-count result below is a prefix of these.
N_MAX_SEEDS = 15
t0 = time.time()
base_cache = {}
for vb in FOLDS:
    tr, va = train[train["batch"] < vb], train[train["batch"] == vb]
    (Xt, Xv), fscale = prep(tr, va)
    base_cache[vb] = dict(
        y=va["gas_class"].values,
        probs=train_seeds(Xt, np.searchsorted(CLASSES, tr["gas_class"].values), Xv,
                          fscale, n_seeds=N_MAX_SEEDS),
    )
    print(f"  fold {vb}: {N_MAX_SEEDS} seeds trained ({len(va)} rows)")
print(f"[{time.time() - t0:.0f}s]")

rows = []
for k in (1, 2, 3, 5, 10, 15):
    per_fold = {vb: f1_of(base_cache[vb]["probs"][:k].mean(0), base_cache[vb]["y"]) for vb in FOLDS}
    # spread across individual seeds, as a measure of how noisy a single run is
    single_sd = np.mean([np.std([f1_of(base_cache[vb]["probs"][s], base_cache[vb]["y"])
                                 for s in range(N_MAX_SEEDS)]) for vb in FOLDS])
    rows.append(dict(seeds=k, mean=np.mean(list(per_fold.values())),
                     large_drift=np.mean([per_fold[b] for b in LARGE_DRIFT]),
                     single_seed_sd=single_sd))
seed_tab = pd.DataFrame(rows)
print("\n=== 1. seed ensembling (nested: k=1 is a subset of k=15) ===")
print(seed_tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\ngain from 3 -> 15 seeds: "
      f"mean {seed_tab.loc[seed_tab.seeds==15,'mean'].iloc[0] - seed_tab.loc[seed_tab.seeds==3,'mean'].iloc[0]:+.4f}"
      f"  large-drift {seed_tab.loc[seed_tab.seeds==15,'large_drift'].iloc[0] - seed_tab.loc[seed_tab.seeds==3,'large_drift'].iloc[0]:+.4f}")

  fold 2: 15 seeds trained (1244 rows)
  fold 3: 15 seeds trained (1586 rows)
  fold 4: 15 seeds trained (161 rows)
  fold 5: 15 seeds trained (197 rows)
  fold 6: 15 seeds trained (2300 rows)
  fold 7: 15 seeds trained (3613 rows)
  fold 8: 15 seeds trained (294 rows)
  fold 9: 15 seeds trained (470 rows)
[274s]

=== 1. seed ensembling (nested: k=1 is a subset of k=15) ===
 seeds   mean  large_drift  single_seed_sd
     1 0.8470       0.8567          0.0165
     2 0.8460       0.8550          0.0165
     3 0.8474       0.8578          0.0165
     5 0.8491       0.8607          0.0165
    10 0.8482       0.8607          0.0165
    15 0.8493       0.8614          0.0165

gain from 3 -> 15 seeds: mean +0.0019  large-drift +0.0036


In [4]:
# 2. Per-sensor gain augmentation, at 15 seeds so the comparison is not seed noise.
print("=== 2. per-sensor gain augmentation ===")
t0 = time.time()
aug_cache = {0.0: base_cache}
for sigma in (0.1, 0.25, 0.5, 1.0):
    cache = {}
    for vb in FOLDS:
        tr, va = train[train["batch"] < vb], train[train["batch"] == vb]
        (Xt, Xv), fscale = prep(tr, va)
        cache[vb] = dict(y=va["gas_class"].values,
                         probs=train_seeds(Xt, np.searchsorted(CLASSES, tr["gas_class"].values),
                                           Xv, fscale, n_seeds=N_MAX_SEEDS, aug_sigma=sigma))
    aug_cache[sigma] = cache
    print(f"  sigma={sigma} done")
print(f"[{time.time() - t0:.0f}s]")

aug_rows = []
for sigma, cache in aug_cache.items():
    pf = {vb: f1_of(cache[vb]["probs"].mean(0), cache[vb]["y"]) for vb in FOLDS}
    aug_rows.append(dict(aug_sigma=sigma, mean=np.mean(list(pf.values())),
                         large_drift=np.mean([pf[b] for b in LARGE_DRIFT]),
                         worst_fold=min(pf.values())))
aug_tab = pd.DataFrame(aug_rows).sort_values("large_drift", ascending=False)
print()
print(aug_tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
BEST_SIGMA = float(aug_tab.iloc[0]["aug_sigma"])
print(f"\nbest aug_sigma by large-drift folds: {BEST_SIGMA}")

=== 2. per-sensor gain augmentation ===
  sigma=0.1 done
  sigma=0.25 done
  sigma=0.5 done
  sigma=1.0 done
[1201s]

 aug_sigma   mean  large_drift  worst_fold
    0.1000 0.8516       0.8628      0.6677
    0.0000 0.8493       0.8614      0.6640
    0.2500 0.8357       0.8386      0.6507
    0.5000 0.8157       0.8132      0.6500
    1.0000 0.8109       0.8098      0.6488

best aug_sigma by large-drift folds: 0.1


In [5]:
# 3. Balanced assignment, on the winning augmentation setting.
def sinkhorn(P, quota, iters=200, eps=1e-12):
    Q = np.clip(P, eps, None).copy()
    for _ in range(iters):
        Q /= Q.sum(1, keepdims=True)
        Q *= (quota / np.maximum(Q.sum(0), eps))
    return Q


def capped_greedy(P, quota):
    """Most-confident-first under hard per-class caps -> exact counts."""
    remaining = np.array(quota, dtype=int).copy()
    out = np.full(len(P), -1, dtype=int)
    for i in np.argsort(-P.max(1)):
        for c in np.argsort(-P[i]):
            if remaining[c] > 0:
                out[i] = c
                remaining[c] -= 1
                break
    return out


RULES = {
    "argmax (unconstrained)": lambda P, q: P.argmax(1),
    "Sinkhorn -> argmax": lambda P, q: sinkhorn(P, q).argmax(1),
    "Sinkhorn + capped greedy (exact)": lambda P, q: capped_greedy(sinkhorn(P, q), q),
}

best_cache = aug_cache[BEST_SIGMA]
rng = np.random.RandomState(0)
R_DRAWS = 20
rr = []
y_all = train["gas_class"].values
for vb in BALANCEABLE:
    va_idx = np.where(train["batch"].values == vb)[0]
    pos = {g: i for i, g in enumerate(va_idx)}
    P_full = best_cache[vb]["probs"].mean(0)
    n = min((y_all[va_idx] == c).sum() for c in CLASSES)
    for _ in range(R_DRAWS):
        pick = np.concatenate([rng.choice(va_idx[y_all[va_idx] == c], n, replace=False)
                               for c in CLASSES])
        rows_i = [pos[i] for i in pick]
        P, y, quota = P_full[rows_i], y_all[pick], np.full(K, n)
        for name, fn in RULES.items():
            rr.append(dict(batch=vb, rule=name,
                           f1=f1_score(y, np.array(CLASSES)[fn(P, quota)], average="macro")))
rules_df = pd.DataFrame(rr)

print(f"=== 3. balanced assignment (folds {[int(b) for b in BALANCEABLE]}, {R_DRAWS} draws each) ===")
piv = rules_df.pivot_table(index="rule", columns="batch", values="f1")
piv["MEAN"] = rules_df.groupby("rule").f1.mean()
print(piv.round(4).sort_values("MEAN", ascending=False).to_string())
base_mean = piv.loc["argmax (unconstrained)", "MEAN"]
print("\ngain over unconstrained argmax:")
for r in piv.index:
    if r != "argmax (unconstrained)":
        print(f"  {r:<36} {piv.loc[r, 'MEAN'] - base_mean:+.4f}")
BEST_RULE = piv["MEAN"].idxmax()
print(f"\nbest rule: {BEST_RULE}")

=== 3. balanced assignment (folds [6, 7, 8, 9], 20 draws each) ===
batch                                  6       7       8       9    MEAN
rule                                                                    
Sinkhorn -> argmax                0.9684  0.9731  0.9569  0.9958  0.9735
Sinkhorn + capped greedy (exact)  0.9592  0.9706  0.9528  1.0000  0.9706
argmax (unconstrained)            0.7721  0.8901  0.9537  0.7638  0.8449

gain over unconstrained argmax:
  Sinkhorn + capped greedy (exact)     +0.1257
  Sinkhorn -> argmax                   +0.1286

best rule: Sinkhorn -> argmax


In [6]:
print("=" * 76)
print("SUMMARY -- effect of each addition, measured independently")
print("=" * 76)
b3 = seed_tab.loc[seed_tab.seeds == 3]
b15 = seed_tab.loc[seed_tab.seeds == 15]
a0 = aug_tab.loc[aug_tab.aug_sigma == 0.0]
print(f"  baseline MLP, 3 seeds            mean={b3['mean'].iloc[0]:.4f}  large-drift={b3['large_drift'].iloc[0]:.4f}")
print(f"  + 15 seeds                       mean={b15['mean'].iloc[0]:.4f}  large-drift={b15['large_drift'].iloc[0]:.4f}"
      f"   ({b15['large_drift'].iloc[0] - b3['large_drift'].iloc[0]:+.4f})")
print(f"  + gain augmentation (s={BEST_SIGMA})    mean={aug_tab.iloc[0]['mean']:.4f}  "
      f"large-drift={aug_tab.iloc[0]['large_drift']:.4f}"
      f"   ({aug_tab.iloc[0]['large_drift'] - a0['large_drift'].iloc[0]:+.4f} vs no aug)")
print(f"  + balanced assignment            {base_mean:.4f} -> {piv['MEAN'].max():.4f}"
      f"   ({piv['MEAN'].max() - base_mean:+.4f}, on balanced folds)")
print("=" * 76)
print("The balanced-assignment row is measured on balanced folds only, so it is not directly")
print("comparable to the other rows -- but it is the only one whose gain does not depend on")
print("the drift direction being predictable from history.")

SUMMARY -- effect of each addition, measured independently
  baseline MLP, 3 seeds            mean=0.8474  large-drift=0.8578
  + 15 seeds                       mean=0.8493  large-drift=0.8614   (+0.0036)
  + gain augmentation (s=0.1)    mean=0.8516  large-drift=0.8628   (+0.0014 vs no aug)
  + balanced assignment            0.8449 -> 0.9735   (+0.1286, on balanced folds)
The balanced-assignment row is measured on balanced folds only, so it is not directly
comparable to the other rows -- but it is the only one whose gain does not depend on
the drift direction being predictable from history.


In [7]:
# Final: refit on all 9 batches with the winning settings, apply the constraint, ship.
print(f"final config: 15 seeds, aug_sigma={BEST_SIGMA}, rule='{BEST_RULE}'")
(X_all, X_te), fscale = prep(train, test)
probs = train_seeds(X_all, np.searchsorted(CLASSES, train["gas_class"].values), X_te,
                    fscale, n_seeds=N_MAX_SEEDS, aug_sigma=BEST_SIGMA)
P = probs.mean(0)

quota = np.full(K, TEST_QUOTA)
pred_free = np.array(CLASSES)[P.argmax(1)]
pred = np.array(CLASSES)[RULES[BEST_RULE](P, quota)]
print(f"the constraint changed {(pred != pred_free).mean():.1%} of the 3600 predictions")

sub = pd.DataFrame({"measurement_id": test["measurement_id"], "gas_class": pred})
assert list(sub.columns) == list(sample_sub.columns)
assert len(sub) == len(sample_sub)
assert (sub["measurement_id"].values == sample_sub["measurement_id"].values).all()
assert sub["gas_class"].isin(range(1, 7)).all() and sub["gas_class"].notna().all()
sub.to_csv("data/submission_mlp_final.csv", index=False)
print("wrote data/submission_mlp_final.csv  (data/submission.csv left untouched)")

vc = sub["gas_class"].value_counts().sort_index()
fv = pd.Series(pred_free).value_counts().sort_index()
print("\nclass counts, unconstrained -> constrained (target 600):")
for c in CLASSES:
    print(f"  class {c}: {fv.get(c, 0):>5} -> {vc.get(c, 0):>5}")
print(f"  deviation from 600: {int((fv.reindex(CLASSES).fillna(0) - 600).abs().sum())} -> "
      f"{int((vc.reindex(CLASSES).fillna(0) - 600).abs().sum())}")

final config: 15 seeds, aug_sigma=0.1, rule='Sinkhorn -> argmax'
the constraint changed 16.2% of the 3600 predictions
wrote data/submission_mlp_final.csv  (data/submission.csv left untouched)

class counts, unconstrained -> constrained (target 600):
  class 1:   450 ->   609
  class 2:   464 ->   583
  class 3:   613 ->   607
  class 4:   471 ->   586
  class 5:   563 ->   604
  class 6:  1039 ->   611
  deviation from 600: 904 -> 62
